# 01 — StateGraph Basics

LangGraph = your agentic loop expressed as an **explicit state machine**.

| Your RISC Strands | LangGraph |
|---|---|
| `@tool` functions | nodes |
| `Agent()` loop | `StateGraph` + edges |
| Agent state dict | `TypedDict` schema |
| Session context | `checkpointer` (thread_id) |

In [ ]:
# Install deps (run once)
# !pip install langgraph langchain langchain-community langchain-ollama

## 1. State

Everything in LangGraph flows through the state. Nodes receive it and return partial updates.

In [ ]:
import operator
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END

class PipelineState(TypedDict):
    date: str
    row_count: int
    # Annotated with operator.add = REDUCER
    # new value is ADDED to existing, not replaced
    log: Annotated[list[str], operator.add]

print("State schema defined")

### Reducers — the thing people miss

```python
# Without reducer: replaces
log: list[str]         # return {"log": ["b"]} → state["log"] == ["b"]

# With reducer: merges
log: Annotated[list[str], operator.add]  # return {"log": ["b"]} → state["log"] == ["a", "b"]
```

`MessagesState` uses `add_messages` reducer — appends and deduplicates by message id.

## 2. Nodes

In [ ]:
def count_rows(state: PipelineState) -> dict:
    count = 75_000
    print(f"[count_rows] {count} rows for {state['date']}")
    return {"row_count": count, "log": [f"counted {count} rows"]}

def full_load(state: PipelineState) -> dict:
    print(f"[full_load] processing {state['row_count']} rows")
    return {"log": ["ran full load"]}

def sample_load(state: PipelineState) -> dict:
    sample = state["row_count"] // 10
    print(f"[sample_load] sampling {sample} rows")
    return {"log": [f"ran sample load ({sample} rows)"]}

## 3. Edges — static and conditional

In [ ]:
def route_by_size(state: PipelineState) -> str:
    """Conditional edge — returns name of next node."""
    return "full_load" if state["row_count"] > 50_000 else "sample_load"

builder = StateGraph(PipelineState)

builder.add_node("count_rows",  count_rows)
builder.add_node("full_load",   full_load)
builder.add_node("sample_load", sample_load)

builder.add_edge(START, "count_rows")                            # static
builder.add_conditional_edges("count_rows", route_by_size)       # dynamic
builder.add_edge("full_load",   END)
builder.add_edge("sample_load", END)

graph = builder.compile()
print("Graph compiled")

In [ ]:
result = graph.invoke({"date": "2024-01-15", "row_count": 0, "log": []})
print(f"\nLog: {result['log']}")

In [ ]:
# Try with small row count — should take sample path
def count_rows_small(state):
    return {"row_count": 10_000, "log": ["counted 10000 rows"]}

builder2 = StateGraph(PipelineState)
builder2.add_node("count_rows",  count_rows_small)
builder2.add_node("full_load",   full_load)
builder2.add_node("sample_load", sample_load)
builder2.add_edge(START, "count_rows")
builder2.add_conditional_edges("count_rows", route_by_size)
builder2.add_edge("full_load",   END)
builder2.add_edge("sample_load", END)

result2 = builder2.compile().invoke({"date": "2024-01-15", "row_count": 0, "log": []})
print(f"Log: {result2['log']}")

## 4. MessagesState + create_react_agent (shortcut for single agent)

In [ ]:
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain_ollama import ChatOllama
from langchain_core.tools import tool

llm = ChatOllama(model="llama3.2:3b")

@tool
def list_tables() -> list[str]:
    """List all available data tables."""
    return ["site_metrics", "labor_events", "pipeline_runs"]

@tool
def get_row_count(table_name: str) -> int:
    """Get number of rows in a table."""
    return {"site_metrics": 300, "labor_events": 300, "pipeline_runs": 50}.get(table_name, -1)

agent = create_react_agent(
    model=llm,
    tools=[list_tables, get_row_count],
    prompt="You are a data quality assistant.",
    checkpointer=InMemorySaver(),
)

config = {"configurable": {"thread_id": "nb-session-1"}}
r = agent.invoke({"messages": [{"role": "user", "content": "what tables do we have?"}]}, config)
print(r["messages"][-1].content)

In [ ]:
# Turn 2 — agent remembers tables from turn 1
r2 = agent.invoke({"messages": [{"role": "user", "content": "how many rows in labor_events?"}]}, config)
print(r2["messages"][-1].content)